In [11]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint, Bounds

## Values
n = 20
d = np.full(n, 225.0)   # demand per household (L/day)
b = np.full(n, 100.0)   # minimum basic water per household (L/day)
S = 3500.0              # total available source water (L/day)

# delivery-loss factor range (terrain/pipe losses)
# Example: a_i in [1.00, 1.25] means 0% to 25% source overhead.
a = np.linspace(1.00, 1.25, n)

# vulnerability priority range (equity weights)
# Higher weight --> shortages for that household are penalized more.
w = np.linspace(1.00, 1.80, n)

#Checks
if n <= 0:
    raise ValueError("n must be positive.")

if np.any(d < b):
    raise ValueError("Demand must be >= minimum basic water for each household.")

if S <= 0:
    raise ValueError("Total supply S must be positive.")

if np.any(a < 1.0):
    raise ValueError("Each delivery factor a_i should be >= 1.0.")

if np.any(w <= 0.0):
    raise ValueError("Each vulnerability weight w_i must be positive.")

#source interval with lower bound b and upper bound d
S_min = np.sum(a * b)
S_max = np.sum(a * d)
print(f"Feasible source interval: [{S_min:.1f}, {S_max:.1f}] L/day")
print(f"Chosen total supply S: {S:.1f} L/day")
print(f"Delivery factor range a: [{np.min(a):.2f}, {np.max(a):.2f}]")
print(f"Vulnerability range w: [{np.min(w):.2f}, {np.max(w):.2f}]")

if S < S_min:
    raise ValueError("Infeasible: supply is below total minimum requirement.")

if S > S_max:
    print("Note: supply is above total demand, some water may remain unused in this model.")

# Objective function 
def objective(x):
    return np.sum(w * (d - x) ** 2)

def objective_grad(x):
    return 2.0 * w * (x - d)

# Constraint and bounds
A = a.reshape(1, -1)
eq_constr = LinearConstraint(A, lb=S, ub=S)

bounds = Bounds(lb=b, ub=d)

# Initial guess 
x0 = np.full(n, S / n)
x0 = np.clip(x0, b, d)


res = minimize(
    fun=objective,
    x0=x0,
    jac=objective_grad,
    method='trust-constr',
    constraints=[eq_constr],
    bounds=bounds,
    options={'verbose': 0}
 )

x_opt = res.x


print(f"Total delivered: {np.sum(x_opt):.2f} L/day")
print(f"Average per household: {np.mean(x_opt):.2f} L/day")
print(f"Households below 100 L/day: {np.sum(x_opt < 100.0)}")

print("\nHousehold allocations (L/day):")
for i in range(min(20, n)):
    print(f"Household {i+1:2d}: {x_opt[i]:.2f}")

Feasible source interval: [2250.0, 5062.5] L/day
Chosen total supply S: 3500.0 L/day
Delivery factor range a: [1.00, 1.25]
Vulnerability range w: [1.00, 1.80]
Total delivered: 3100.73 L/day
Average per household: 155.04 L/day
Households below 100 L/day: 0

Household allocations (L/day):
Household  1: 139.59
Household  2: 141.97
Household  3: 144.16
Household  4: 146.18
Household  5: 148.06
Household  6: 149.81
Household  7: 151.44
Household  8: 152.96
Household  9: 154.39
Household 10: 155.73
Household 11: 156.99
Household 12: 158.18
Household 13: 159.30
Household 14: 160.37
Household 15: 161.37
Household 16: 162.32
Household 17: 163.23
Household 18: 164.09
Household 19: 164.91
Household 20: 165.69
